In [1]:
# [셀 0] 필수 설치 (한 번만)
!pip -q install -U transformers datasets evaluate sentencepiece accelerate Korpora rouge_score absl-py


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 51.8 MB/s eta 0:00:00


In [2]:
# [셀 1] 공통 설정 (FAST 모드)
import os, random
import numpy as np
import torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

SAVE_DIR = "/content/models"
os.makedirs(SAVE_DIR, exist_ok=True)

# ---- FAST 하이퍼파라미터 (필요하면 여기만 조절) ----
# (분류: BERT/KoELECTRA)
CLS_N_SAMPLES = 12000
CLS_MAX_LEN   = 128
CLS_BATCH     = 16
CLS_EPOCHS    = 2
CLS_MAX_STEPS = 300          # epoch당 최대 배치 수(컷) / None이면 전체
CLS_LR        = 2e-5

# (요약: BART/T5)
SUM_N_SAMPLES = 1200
SUM_SRC_LEN   = 256
SUM_TGT_LEN   = 64
BART_BATCH    = 4
T5_BATCH      = 8
SUM_EPOCHS    = 1
SUM_MAX_STEPS = 150
SUM_LR        = 5e-5


device: cuda
GPU: Tesla T4


In [3]:
# [셀 2] (예제 7.15) 데이터 로드/샘플링/분할
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load("nsmc")

df = pd.DataFrame(corpus.test).sample(CLS_N_SAMPLES, random_state=42).reset_index(drop=True)

n = len(df)
train_df = df.iloc[: int(0.6*n)].reset_index(drop=True)
valid_df = df.iloc[int(0.6*n): int(0.8*n)].reset_index(drop=True)
test_df  = df.iloc[int(0.8*n):].reset_index(drop=True)

print(train_df.head().to_markdown())
print("sizes:", len(train_df), len(valid_df), len(test_df))



    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 120MB/s]                             
[nsmc] download ratings_test.txt: 4.90MB [00:00, 88.1MB/s]


|    | text                                                                                     |   label |
|---:|:-----------------------------------------------------------------------------------------|--------:|
|  0 | 모든 편견을 날려 버리는 가슴 따뜻한 영화. 로버트 드 니로, 필립 세이모어 호프만 영원하라. |       1 |
|  1 | 무한 리메이크의 소재. 감독의 역량은 항상 그 자리에...                                    |       0 |
|  2 | 신날 것 없는 애니.                                                                       |       0 |
|  3 | 잔잔 격동                                                                                |       1 |
|  4 | 오랜만에 찾은 주말의 명화의 보석                                                         |       1 |
sizes: 7200 2400 2400


In [4]:
# [셀 3] (예제 7.15) 토크나이즈 + DataLoader (CPU 텐서로 만들고 배치만 GPU로)
from transformers import BertTokenizerFast
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler

tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased", do_lower_case=False)

def make_cls_dataset(df_):
    enc = tokenizer(
        df_["text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=CLS_MAX_LEN,
        return_tensors="pt",
    )
    labels = torch.tensor(df_["label"].values, dtype=torch.long)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)

def make_loader(dataset, train=True, batch_size=16):
    sampler = RandomSampler(dataset) if train else SequentialSampler(dataset)
    return DataLoader(
        dataset,
        sampler=sampler,
        batch_size=batch_size,
        pin_memory=(device.type == "cuda"),
        num_workers=2,
    )

train_loader = make_loader(make_cls_dataset(train_df), train=True,  batch_size=CLS_BATCH)
valid_loader = make_loader(make_cls_dataset(valid_df), train=False, batch_size=CLS_BATCH)
test_loader  = make_loader(make_cls_dataset(test_df),  train=False, batch_size=CLS_BATCH)

batch = next(iter(train_loader))
print([x.shape for x in batch])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

[torch.Size([16, 128]), torch.Size([16, 128]), torch.Size([16])]


In [5]:
# [셀 4] (예제 7.16) BERT 학습 (AMP + max_steps 컷) + 저장
from transformers import BertForSequenceClassification
from torch import optim
from torch.cuda.amp import autocast, GradScaler

model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=2
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=CLS_LR, eps=1e-8)
scaler = GradScaler(enabled=(device.type == "cuda"))

def train_cls_one_epoch(model, loader, max_steps=None):
    model.train()
    total_loss = 0.0
    steps = 0

    for step, (input_ids, attention_mask, labels) in enumerate(loader):
        if max_steps is not None and step >= max_steps:
            break

        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=(device.type == "cuda")):
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        steps += 1

    return total_loss / max(steps, 1)

@torch.no_grad()
def eval_cls(model, loader, max_steps=None):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    steps = 0

    for step, (input_ids, attention_mask, labels) in enumerate(loader):
        if max_steps is not None and step >= max_steps:
            break

        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = out.loss
        preds = out.logits.argmax(dim=-1)

        total_loss += loss.item()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        steps += 1

    return total_loss / max(steps, 1), correct / max(total, 1)

best_loss = float("inf")
ckpt_path = os.path.join(SAVE_DIR, "BertForSequenceClassification.pt")

for epoch in range(CLS_EPOCHS):
    tr_loss = train_cls_one_epoch(model, train_loader, max_steps=CLS_MAX_STEPS)
    va_loss, va_acc = eval_cls(model, valid_loader)
    print(f"Epoch {epoch+1}: train {tr_loss:.4f} | val {va_loss:.4f} | acc {va_acc:.4f}")

    if va_loss < best_loss:
        best_loss = va_loss
        torch.save(model.state_dict(), ckpt_path)
        print("Saved ->", ckpt_path)


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1139383725.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device.type == "cuda"))
/tmp/ipython-input-1139383725.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):


Epoch 1: train 0.5959 | val 0.5205 | acc 0.7429
Saved -> /content/models/BertForSequenceClassification.pt
Epoch 2: train 0.4531 | val 0.4530 | acc 0.7883
Saved -> /content/models/BertForSequenceClassification.pt


In [6]:
# [셀 5] (예제 7.17) 저장 모델 로드 후 Test 평가
model2 = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased",
    num_labels=2
).to(device)

state = torch.load(os.path.join(SAVE_DIR, "BertForSequenceClassification.pt"), map_location=device)
model2.load_state_dict(state)

te_loss, te_acc = eval_cls(model2, test_loader)
print(f"Test Loss: {te_loss:.4f}")
print(f"Test Acc : {te_acc:.4f}")

del model2
torch.cuda.empty_cache()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Test Loss: 0.4592
Test Acc : 0.7817


In [7]:
# [셀 6] (예제 7.18) 데이터 로드/샘플링/분할
from datasets import load_dataset

news = load_dataset("argilla/news-summary", split="test")
df = news.to_pandas().sample(SUM_N_SAMPLES, random_state=42)[["text", "prediction"]].reset_index(drop=True)
df["prediction"] = df["prediction"].map(lambda x: x[0]["text"])

n = len(df)
train_df = df.iloc[: int(0.6*n)].reset_index(drop=True)
valid_df = df.iloc[int(0.6*n): int(0.8*n)].reset_index(drop=True)
test_df  = df.iloc[int(0.8*n):].reset_index(drop=True)

print("news:", train_df["text"].iloc[0][:200])
print("sum :", train_df["prediction"].iloc[0][:80])
print("sizes:", len(train_df), len(valid_df), len(test_df))


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-ebc48879f34571(…):   0%|          | 0.00/1.54M [00:00<?, ?B/s]

data/test-00000-of-00001-6227bd8eb10a9b5(…):   0%|          | 0.00/31.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20417 [00:00<?, ? examples/s]

news: WASHINGTON (Reuters) - President Barack Obama did not specify a candidate preference in the race for the Democratic presidential nomination at a Democratic National Committee fundraising event, White 
sum : Obama did not indicate preference for Democratic candidate: White House
sizes: 720 240 240


In [8]:
# [셀 7] (예제 7.19) 토크나이즈 + DataLoader (max_length 줄임, 배치만 GPU)
from transformers import BartTokenizerFast
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

bart_tok = BartTokenizerFast.from_pretrained("facebook/bart-base")

def make_sum_dataset(df_):
    enc = bart_tok(
        df_["text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=SUM_SRC_LEN,
        return_tensors="pt",
    )
    with bart_tok.as_target_tokenizer():
        dec = bart_tok(
            df_["prediction"].tolist(),
            padding="max_length",
            truncation=True,
            max_length=SUM_TGT_LEN,
            return_tensors="pt",
        )
    labels = dec["input_ids"]
    labels[labels == bart_tok.pad_token_id] = -100
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)

def make_sum_loader(dataset, train=True, batch_size=4):
    sampler = RandomSampler(dataset) if train else SequentialSampler(dataset)
    return DataLoader(dataset, sampler=sampler, batch_size=batch_size,
                      pin_memory=(device.type=="cuda"), num_workers=2)

bart_train_loader = make_sum_loader(make_sum_dataset(train_df), train=True,  batch_size=BART_BATCH)
bart_valid_loader = make_sum_loader(make_sum_dataset(valid_df), train=False, batch_size=BART_BATCH)
bart_test_loader  = make_sum_loader(make_sum_dataset(test_df),  train=False, batch_size=BART_BATCH)

print([x.shape for x in next(iter(bart_train_loader))])


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


[torch.Size([4, 256]), torch.Size([4, 256]), torch.Size([4, 64])]


In [9]:
# [셀 8] (예제 7.20~7.22) BART 학습 (AMP + max_steps 컷) + 저장
from transformers import BartForConditionalGeneration
from torch import optim
from torch.cuda.amp import autocast, GradScaler

bart = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(device)
bart_opt = optim.AdamW(bart.parameters(), lr=SUM_LR, eps=1e-8)
bart_scaler = GradScaler(enabled=(device.type == "cuda"))

def train_sum_one_epoch(model, loader, optimizer, max_steps=None):
    model.train()
    total = 0.0
    steps = 0

    for step, (input_ids, attention_mask, labels) in enumerate(loader):
        if max_steps is not None and step >= max_steps:
            break

        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=(device.type == "cuda")):
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss

        bart_scaler.scale(loss).backward()
        bart_scaler.step(optimizer)
        bart_scaler.update()

        total += loss.item()
        steps += 1

    return total / max(steps, 1)

@torch.no_grad()
def eval_sum_loss(model, loader, max_batches=10):
    model.eval()
    total = 0.0
    steps = 0
    for step, (input_ids, attention_mask, labels) in enumerate(loader):
        if step >= max_batches:
            break
        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total += out.loss.item()
        steps += 1
    return total / max(steps, 1)

bart_best = float("inf")
bart_ckpt = os.path.join(SAVE_DIR, "BartForConditionalGeneration.pt")

for epoch in range(SUM_EPOCHS):
    tr = train_sum_one_epoch(bart, bart_train_loader, bart_opt, max_steps=SUM_MAX_STEPS)
    va = eval_sum_loss(bart, bart_valid_loader, max_batches=10)
    print(f"Epoch {epoch+1}: train {tr:.4f} | val {va:.4f}")

    if va < bart_best:
        bart_best = va
        torch.save(bart.state_dict(), bart_ckpt)
        print("Saved ->", bart_ckpt)


model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

/tmp/ipython-input-693860061.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  bart_scaler = GradScaler(enabled=(device.type == "cuda"))
/tmp/ipython-input-693860061.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):


Epoch 1: train 2.3719 | val 1.9507
Saved -> /content/models/BartForConditionalGeneration.pt


In [10]:
# [셀 9] (예제 7.23) ROUGE-2(빠른 샘플 평가) + 몇 개 생성 비교
import evaluate

rouge = evaluate.load("rouge")

@torch.no_grad()
def quick_rouge2(model, loader, max_batches=5):
    model.eval()
    preds, refs = [], []

    for b, (input_ids, attention_mask, labels) in enumerate(loader):
        if b >= max_batches:
            break

        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels = labels.clone()
        labels[labels == -100] = bart_tok.pad_token_id

        gen = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_beams=2,
            max_new_tokens=SUM_TGT_LEN,
        )

        preds += bart_tok.batch_decode(gen, skip_special_tokens=True)
        refs  += bart_tok.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(predictions=preds, references=refs)
    return scores["rouge2"], preds[:3], refs[:3]

# 저장본 로드 후 평가
bart2 = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(device)
bart2.load_state_dict(torch.load(bart_ckpt, map_location=device))

r2, p3, g3 = quick_rouge2(bart2, bart_test_loader, max_batches=5)
print("Quick ROUGE-2:", r2)
for i in range(3):
    print("\n[정답]\n", g3[i])
    print("[모델]\n", p3[i])

del bart2
torch.cuda.empty_cache()


Quick ROUGE-2: 0.1931295433153018

[정답]
 No word Tuesday on Supreme Court nomination: White House
[모델]
 White House does not plan announcement on Supreme Court nominee

[정답]
 Former Trump campaign staffer files discrimination complaint: NYT
[모델]
 Ex-Trump campaign staffer says campaign discrimination

[정답]
 China calls for restraint when asked about North Korea hydrogen bomb threat
[모델]
 China calls on restraint after North Korea's foreign minister quoted as saying he wants hydrogen bomb test


In [11]:
# [셀 10] (옵션) pipeline로 요약 보기 (GPU면 device=0)
from transformers import pipeline

pipe_device = 0 if device.type == "cuda" else -1

bart_pipe_model = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(device)
bart_pipe_model.load_state_dict(torch.load(bart_ckpt, map_location=device))

summarizer = pipeline(
    "summarization",
    model=bart_pipe_model,
    tokenizer=bart_tok,
    device=pipe_device
)

for i in range(3):
    txt = test_df["text"].iloc[i]
    gold = test_df["prediction"].iloc[i]
    pred = summarizer(txt, max_new_tokens=SUM_TGT_LEN, num_beams=2)[0]["summary_text"]
    print("\n정답:", gold)
    print("모델:", pred)

del bart_pipe_model
torch.cuda.empty_cache()


Device set to use cuda:0
Your max_length is set to 128, but your input_length is only 105. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=52)
Your max_length is set to 128, but your input_length is only 100. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=50)



정답: No word Tuesday on Supreme Court nomination: White House
모델: White House does not plan announcement on Supreme Court nominee

정답: Former Trump campaign staffer files discrimination complaint: NYT
모델: Ex-Trump campaign staffer says campaign discriminated against her: New York Times

정답: China calls for restraint when asked about North Korea hydrogen bomb threat
모델: China calls on restraint after North Korea's foreign minister quoted as saying he wants hydrogen bomb test


In [12]:
# [셀 11] (예제 7.24) 데이터 로드/샘플링/분할
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load("nsmc")
df = pd.DataFrame(corpus.test).sample(CLS_N_SAMPLES, random_state=42).reset_index(drop=True)

n = len(df)
train_df = df.iloc[: int(0.6*n)].reset_index(drop=True)
valid_df = df.iloc[int(0.6*n): int(0.8*n)].reset_index(drop=True)
test_df  = df.iloc[int(0.8*n):].reset_index(drop=True)

print("sizes:", len(train_df), len(valid_df), len(test_df))



    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

In [13]:
# [셀 12] (예제 7.24) 토크나이즈 + DataLoader
from transformers import ElectraTokenizerFast
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

elec_tok = ElectraTokenizerFast.from_pretrained("monologg/koelectra-base-v3-discriminator", do_lower_case=False)

def make_elec_dataset(df_):
    enc = elec_tok(
        df_["text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=CLS_MAX_LEN,
        return_tensors="pt",
    )
    labels = torch.tensor(df_["label"].values, dtype=torch.long)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)

def make_elec_loader(dataset, train=True):
    sampler = RandomSampler(dataset) if train else SequentialSampler(dataset)
    return DataLoader(dataset, sampler=sampler, batch_size=CLS_BATCH,
                      pin_memory=(device.type=="cuda"), num_workers=2)

elec_train_loader = make_elec_loader(make_elec_dataset(train_df), train=True)
elec_valid_loader = make_elec_loader(make_elec_dataset(valid_df), train=False)
elec_test_loader  = make_elec_loader(make_elec_dataset(test_df),  train=False)


tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

In [14]:
# [셀 13] (예제 7.25) KoELECTRA 학습 + 저장
from transformers import ElectraForSequenceClassification
from torch import optim
from torch.cuda.amp import autocast, GradScaler

elec = ElectraForSequenceClassification.from_pretrained(
    "monologg/koelectra-base-v3-discriminator",
    num_labels=2
).to(device)

elec_opt = optim.AdamW(elec.parameters(), lr=CLS_LR, eps=1e-8)
elec_scaler = GradScaler(enabled=(device.type=="cuda"))

def train_cls_one_epoch_amp(model, loader, optimizer, scaler, max_steps=None):
    model.train()
    total = 0.0
    steps = 0
    for step, (input_ids, attention_mask, labels) in enumerate(loader):
        if max_steps is not None and step >= max_steps:
            break
        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type=="cuda")):
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total += loss.item()
        steps += 1
    return total / max(steps, 1)

@torch.no_grad()
def eval_cls_simple(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    steps = 0
    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += out.loss.item()
        correct += (out.logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)
        steps += 1
    return total_loss / max(steps, 1), correct / max(total, 1)

elec_best = float("inf")
elec_ckpt = os.path.join(SAVE_DIR, "ElectraForSequenceClassification.pt")

for epoch in range(CLS_EPOCHS):
    tr = train_cls_one_epoch_amp(elec, elec_train_loader, elec_opt, elec_scaler, max_steps=CLS_MAX_STEPS)
    va_loss, va_acc = eval_cls_simple(elec, elec_valid_loader)
    print(f"Epoch {epoch+1}: train {tr:.4f} | val {va_loss:.4f} | acc {va_acc:.4f}")
    if va_loss < elec_best:
        elec_best = va_loss
        torch.save(elec.state_dict(), elec_ckpt)
        print("Saved ->", elec_ckpt)


pytorch_model.bin:   0%|          | 0.00/452M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1639261837.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  elec_scaler = GradScaler(enabled=(device.type=="cuda"))


model.safetensors:   0%|          | 0.00/452M [00:00<?, ?B/s]

/tmp/ipython-input-1639261837.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type=="cuda")):


Epoch 1: train 0.4603 | val 0.3322 | acc 0.8662
Saved -> /content/models/ElectraForSequenceClassification.pt
Epoch 2: train 0.2930 | val 0.3155 | acc 0.8679
Saved -> /content/models/ElectraForSequenceClassification.pt


In [15]:
# [셀 14] Test 평가
elec2 = ElectraForSequenceClassification.from_pretrained(
    "monologg/koelectra-base-v3-discriminator",
    num_labels=2
).to(device)
elec2.load_state_dict(torch.load(elec_ckpt, map_location=device))

te_loss, te_acc = eval_cls_simple(elec2, elec_test_loader)
print(f"Test Loss: {te_loss:.4f}")
print(f"Test Acc : {te_acc:.4f}")

del elec2
torch.cuda.empty_cache()


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Test Loss: 0.3687
Test Acc : 0.8512


In [16]:
# [셀 15] (예제 7.26) 데이터 로드/샘플링/분할 + prefix
from datasets import load_dataset

news = load_dataset("argilla/news-summary", split="test")
df = news.to_pandas().sample(SUM_N_SAMPLES, random_state=42)[["text", "prediction"]].reset_index(drop=True)
df["text"] = "summarize: " + df["text"]
df["prediction"] = df["prediction"].map(lambda x: x[0]["text"])

n = len(df)
train_df = df.iloc[: int(0.6*n)].reset_index(drop=True)
valid_df = df.iloc[int(0.6*n): int(0.8*n)].reset_index(drop=True)
test_df  = df.iloc[int(0.8*n):].reset_index(drop=True)

print("src:", train_df["text"].iloc[0][:200])
print("tgt:", train_df["prediction"].iloc[0][:80])
print("sizes:", len(train_df), len(valid_df), len(test_df))


src: summarize: WASHINGTON (Reuters) - President Barack Obama did not specify a candidate preference in the race for the Democratic presidential nomination at a Democratic National Committee fundraising ev
tgt: Obama did not indicate preference for Democratic candidate: White House
sizes: 720 240 240


In [17]:
# [셀 16] (예제 7.27) 토크나이즈 + DataLoader (CPU → 배치만 GPU)
from transformers import T5TokenizerFast
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

t5_tok = T5TokenizerFast.from_pretrained("t5-small")

def make_t5_dataset(df_):
    src = t5_tok(
        df_["text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=SUM_SRC_LEN,
        return_tensors="pt",
    )
    tgt = t5_tok(
        df_["prediction"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=SUM_TGT_LEN,
        return_tensors="pt",
    )
    labels = tgt["input_ids"]
    labels[labels == t5_tok.pad_token_id] = -100
    return TensorDataset(src["input_ids"], src["attention_mask"], labels)

def make_t5_loader(dataset, train=True, batch_size=8):
    sampler = RandomSampler(dataset) if train else SequentialSampler(dataset)
    return DataLoader(dataset, sampler=sampler, batch_size=batch_size,
                      pin_memory=(device.type=="cuda"), num_workers=2)

t5_train_loader = make_t5_loader(make_t5_dataset(train_df), train=True,  batch_size=T5_BATCH)
t5_valid_loader = make_t5_loader(make_t5_dataset(valid_df), train=False, batch_size=T5_BATCH)
t5_test_loader  = make_t5_loader(make_t5_dataset(test_df),  train=False, batch_size=T5_BATCH)

print([x.shape for x in next(iter(t5_train_loader))])


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

[torch.Size([8, 256]), torch.Size([8, 256]), torch.Size([8, 64])]


In [18]:
# [셀 17] (예제 7.28~7.29) T5 학습 (labels만 주면 내부에서 shift 처리) + 저장
from transformers import T5ForConditionalGeneration
from torch import optim
from torch.cuda.amp import autocast, GradScaler

t5 = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
t5_opt = optim.AdamW(t5.parameters(), lr=1e-5, eps=1e-8)  # 원본 예제와 비슷하게
t5_scaler = GradScaler(enabled=(device.type=="cuda"))

def train_t5_one_epoch(model, loader, optimizer, scaler, max_steps=None):
    model.train()
    total = 0.0
    steps = 0
    for step, (input_ids, attention_mask, labels) in enumerate(loader):
        if max_steps is not None and step >= max_steps:
            break

        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device.type=="cuda")):
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total += loss.item()
        steps += 1
    return total / max(steps, 1)

@torch.no_grad()
def eval_t5_loss(model, loader, max_batches=10):
    model.eval()
    total = 0.0
    steps = 0
    for b, (input_ids, attention_mask, labels) in enumerate(loader):
        if b >= max_batches:
            break
        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total += out.loss.item()
        steps += 1
    return total / max(steps, 1)

t5_best = float("inf")
t5_ckpt = os.path.join(SAVE_DIR, "T5ForConditionalGeneration.pt")

for epoch in range(SUM_EPOCHS):
    tr = train_t5_one_epoch(t5, t5_train_loader, t5_opt, t5_scaler, max_steps=SUM_MAX_STEPS)
    va = eval_t5_loss(t5, t5_valid_loader, max_batches=10)
    print(f"Epoch {epoch+1}: train {tr:.4f} | val {va:.4f}")
    if va < t5_best:
        t5_best = va
        torch.save(t5.state_dict(), t5_ckpt)
        print("Saved ->", t5_ckpt)


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/tmp/ipython-input-1894600592.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  t5_scaler = GradScaler(enabled=(device.type=="cuda"))
/tmp/ipython-input-1894600592.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type=="cuda")):


Epoch 1: train 3.8316 | val 3.3290
Saved -> /content/models/T5ForConditionalGeneration.pt


In [19]:
# [셀 18] (예제 7.30) 생성 결과 확인 (몇 개만)
t5_gen = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
t5_gen.load_state_dict(torch.load(t5_ckpt, map_location=device))
t5_gen.eval()

with torch.no_grad():
    input_ids, attention_mask, labels = next(iter(t5_test_loader))
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)

    gen = t5_gen.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        num_beams=2,
        max_new_tokens=SUM_TGT_LEN,
        repetition_penalty=2.0,
    )

    # labels 디코딩용 복원
    labels = labels.clone()
    labels[labels == -100] = t5_tok.pad_token_id

    preds = t5_tok.batch_decode(gen, skip_special_tokens=True)
    refs  = t5_tok.batch_decode(labels, skip_special_tokens=True)

for i in range(3):
    print("\nGenerated:", preds[i])
    print("Actual   :", refs[i])



Generated: the white house does not plan any announcement on a nominee for the Supreme Court. president Barack Obama has said he plans to soon present a nominee to fill the vacancy left by the Feb. 13 death of conservative Justice Antonin Scalia.
Actual   : No word Tuesday on Supreme Court nomination: White House

Generated: Elizabeth Mae Davidson filed complaint with the Iowa Civil Rights Commission. she claimed female staffers were paid less than male staffers, report says. Davidson also claimed that Trump addressed her and a young female volunteer with a remark.
Actual   : Former Trump campaign staffer files discrimination complaint: NYT

Generated: China calls on all parties to exercise restraint after north Korea's foreign minister quoted as saying he believes the North could consider conducting a hydrogen bomb test in the Pacific Ocean. his comments came after U.S. president Donald Trump ordered new sanctions against North Korea on Thursday over its nuclear and missile programs
